In [1]:
from pathlib import Path
import os
print(os.getcwd())
print('\n'.join(str(p) for p in Path('/workspace').iterdir()))

/workspace
/workspace/fit_corrosion.py
/workspace/memory
/workspace/.config
/workspace/copper-anchor.md
/workspace/phys_tables.json
/workspace/corrosion_data.csv
/workspace/corrosion_fit.json
/workspace/calphad_grid.csv
/workspace/.skills
/workspace/corrosion_fit.md
/workspace/lost+found
/workspace/.kernel_llm_logs_1.jsonl
/workspace/.prompts
/workspace/truth_phys.py
/workspace/bofn_alloy.py
/workspace/phys_tables.md


In [2]:
import pandas as pd, numpy as np, json, asyncio, requests
from pathlib import Path
OUT=Path('/workspace/outputs'); OUT.mkdir(exist_ok=True)
(OUT/'sources').mkdir(exist_ok=True)
df=pd.read_csv('/workspace/corrosion_data.csv')
print(df.groupby('study').agg(n=('rate','size'),weight=('weight','first')))
print(pd.read_csv('/workspace/calphad_grid.csv').head().to_string(index=False))
print('Plan: independently fit weighted within-study centered regressions; globally minimize Ni with piecewise search limits; audit sources; separately execute truth and test implementation against the stated dense reference.')
from e14c.web import web_search, fetch_paper, cite_doi, save_citation
queries=['Ravindran 1983 Copper Nickel Alloys Marine Aquacultural Engineering pdf','Banning Oden 1973 Corrosion resistance metals hot brines 8601 pdf','Al Fozan Malik 2008 Effect seawater level corrosion 0.145 0.067 pdf']
searches=await asyncio.gather(*(web_search(q,num_results=4) for q in queries))
for q,ss in zip(queries,searches):
 print('\n',q)
 for s in ss: print(s['title'],s['url'])
(OUT/'sources/search_results.json').write_text(json.dumps(searches,indent=2))

                       n  weight
study                           
AlFozan2008            2     1.0
CohenWhitted1971       5     0.5
LewisLaQue1981_1y      4     0.5
LewisLaQue1981_quiet   2     1.0
Qian2025               2     1.0
Ravindran1983          3     1.0
Tan2024                4     1.0
Wheatfall1967_123d     2     0.5
Wheatfall1967_surface  2     0.5
Yang2018               2     1.0
Zhao2022               2     0.5
 Ni_wt  Al_wt  Fe_wt  Mn_wt  liquidus_C  solidus_C  range_C     primary  liquid_sol  alpha_sol  beta_sol  kappa_fe_sol  fcc_feni_sol  nial_sol  gprime_sol  other_sol  liquid_900  alpha_900  beta_900  kappa_fe_900  fcc_feni_900  nial_900  gprime_900  other_900  liquid_600  alpha_600  beta_600  kappa_fe_600  fcc_feni_600  nial_600  gprime_600  other_600  alpha600_xni  alpha600_xal  alpha600_xfe  alpha600_xmn  ok  wall_s
   0.0    0.0    0.0    0.0     1084.51    1084.51     0.00 alpha (fcc)     1.00000    0.00000       0.0           0.0           0.0       0.0       


 Ravindran 1983 Copper Nickel Alloys Marine Aquacultural Engineering pdf
 https://epubs.icar.org.in/index.php/FT/article/download/70952/60283/468475
Copper-nickel alloys for marine aquacultural engineering https://doi.org/10.56093/ft.v20i2.70952
COPPER - NICKEL ALLOYS FOR MARINE AQUA CULTURAL ENGINEERING
							| Fishery Technology https://epubs.icar.org.in/index.php/FT/article/view/70952
Holdings:  :: Library Catalog https://info.umar.mx/Record/aquadocs_http___hdl_handle_net_1834_33802

 Banning Oden 1973 Corrosion resistance metals hot brines 8601 pdf
in Hot Brines: A Literature Review https://stacks.cdc.gov/view/cdc/235753/cdc_235753_DS1.pdf
Catalog Record: Corrosion resistance of metals in hot brines... | HathiTrust Digital Library https://catalog.hathitrust.org/Record/005979181
Corrosion resistance of metals in hot brines : a literature review / http://hdl.handle.net/2027/mdp.39015078516294
Corrosion resistance of metals in hot brines: a literature review. [Literature review thro

54868

In [3]:
from scipy.optimize import lsq_linear, minimize, differential_evolution, brentq
from scipy.interpolate import RegularGridInterpolator
BASE=dict(ni_scale=5.,ni_form='sat',al_form='quad',fe_sol=1.5,ridge=.05,signs=True)
def feature(c,cfg):
 c=np.asarray(c); n,a,f=np.moveaxis(c,-1,0)
 return np.stack([(-np.expm1(-n/cfg['ni_scale']) if cfg['ni_form']=='sat' else n/10),a,(a*a/10 if cfg['al_form']=='quad' else np.zeros_like(a)),np.minimum(f,cfg['fe_sol'])],axis=-1)
def own_fit(data=df,**opts):
 cfg={**BASE,**opts}; X=feature(data[['ni','al','fe']].to_numpy(),cfg); y=np.log(data.rate.to_numpy()); w=data.weight.to_numpy()
 # Eliminate nuisance intercepts analytically using weighted within-study means.
 xc=X.copy(); yc=y.copy()
 for s in data.study.unique():
  ii=(data.study==s).to_numpy(); xc[ii]-=np.average(X[ii],axis=0,weights=w[ii]); yc[ii]-=np.average(y[ii],weights=w[ii])
 A=np.vstack([xc*np.sqrt(w[:,None]),np.sqrt(cfg['ridge'])*np.eye(4)]); z=np.r_[yc*np.sqrt(w),np.zeros(4)]
 hi=np.array([0,np.inf,np.inf,0]) if cfg['signs'] else np.full(4,np.inf)
 beta=lsq_linear(A,z,bounds=(-np.full(4,np.inf),hi),tol=1e-13,max_iter=500).x
 res=yc-xc@beta
 return dict(cfg=cfg,beta=beta,rmse=np.sqrt(np.mean(res**2)),n=len(data),rank=np.linalg.matrix_rank(xc),objective=float(sum(w*res**2)+cfg['ridge']*sum(beta**2)))
base=own_fit(); print(base)
saved=json.loads(Path('/workspace/corrosion_fit.json').read_text())
print('Saved coefficient differences',base['beta']-np.array([saved['coef'][s] for s in ['ni','al','al2','fe']]))
grid=pd.read_csv('/workspace/calphad_grid.csv'); print('Grid failed:',(grid.ok!=1).sum(),'missing liquidus:',grid.liquidus_C.isna().sum())
print('Axes:',[sorted(grid[c].unique()) for c in ['Ni_wt','Al_wt','Fe_wt','Mn_wt']])
print('Mn liquidus differences:',grid.groupby(['Ni_wt','Al_wt','Fe_wt']).liquidus_C.apply(lambda z:z.iloc[-1]-z.iloc[0]).describe())
axes=[np.sort(grid[c].unique()) for c in ['Ni_wt','Al_wt','Fe_wt','Mn_wt']]
liquid=grid.liquidus_C.fillna(1550).to_numpy().reshape(8,6,6,3)
L=RegularGridInterpolator(axes,liquid,bounds_error=False,fill_value=None)
print('Reference L',L([[10,0,1.4,0],[30,0,.6,0]]))
# Check whether any axis-cell would make Mn=0 minimize liquidus: the uploaded claim is not assumed.
delta=np.diff(liquid,axis=3)
print('Liquidus Mn incremental differences min/max',delta.min(),delta.max(),'; Mn=0 lower than both other nodes:',np.sum(liquid[...,0]<=liquid.min(axis=3)+1e-8),'of',288)
np.savez(OUT/'independent_base.npz',beta=base['beta'])

{'cfg': {'ni_scale': 5.0, 'ni_form': 'sat', 'al_form': 'quad', 'fe_sol': 1.5, 'ridge': 0.05, 'signs': True}, 'beta': array([-8.40920821e-01, -2.79937945e-01,  3.85670330e-01, -1.11539278e-18]), 'rmse': np.float64(0.22669595203167767), 'n': 30, 'rank': np.int64(4), 'objective': 1.0919894257345708}
Saved coefficient differences [-1.57115432e-11 -1.43851597e-12 -8.53261906e-13  2.83416927e-11]
Grid failed: 0 missing liquidus: 33
Axes: [[np.float64(0.0), np.float64(4.2857), np.float64(8.5714), np.float64(12.8571), np.float64(17.1429), np.float64(21.4286), np.float64(25.7143), np.float64(30.0)], [np.float64(0.0), np.float64(1.9), np.float64(3.8), np.float64(5.7), np.float64(7.6), np.float64(9.5)], [np.float64(0.0), np.float64(0.9), np.float64(1.8), np.float64(2.7), np.float64(3.6), np.float64(4.5)], [np.float64(0.0), np.float64(2.0), np.float64(4.0)]]
Mn liquidus differences: count    275.000000
mean     -21.715091
std       11.387512
min      -36.880000
25%      -31.870000
50%      -23.430

In [4]:
# This session has no project ID; use a local citation namespace (not a remote project).
os.environ.setdefault('PROJECT_ID','local-copper-review')
from e14c.web import citation_files
cites=await asyncio.gather(*(cite_doi(d) for d in dois[1:]),return_exceptions=True)
for c in cites: print(str(c)[:1200])
print(save_citation('''@online{banning1973corrosion,
 author={Banning, L. H. and Oden, L. L.},
 title={Corrosion Resistance of Metals in Hot Brines: A Literature Review},
 date={1973}, institution={U.S. Bureau of Mines}, number={Information Circular 8601},
 url={https://stacks.cdc.gov/view/cdc/235753/cdc_235753_DS1.pdf}, urldate={2026-10-01}
}'''))
print(save_citation('''@article{ravindran1983copper,
 author={Ravindran, K.}, title={Copper-Nickel Alloys for Marine Aquacultural Engineering},
 journal={Fishery Technology}, volume={20}, number={2}, pages={99--105}, year={1983},
 url={https://epubs.icar.org.in/index.php/FT/article/download/70952/60283/468475}
}'''))

@book{1991, title={Survey of degradation modes of four nickel-chromium-molybdenum alloys}, url={http://dx.doi.org/10.2172/138275}, DOI={10.2172/138275}, institution={Office of Scientific and Technical Information (OSTI)}, author={ and Gdowski, G and }, year={1991}, month=Mar }
@article{Al_Fozan_2008, title={Effect of seawater level on corrosion behavior of different alloys}, volume={228}, ISSN={0011-9164}, url={http://dx.doi.org/10.1016/j.desal.2007.08.007}, DOI={10.1016/j.desal.2007.08.007}, number={1-3}, journal={Desalination}, publisher={Elsevier BV}, author={Al-Fozan, Saleh A. and Malik, Anees U.}, year={2008}, month=Aug, pages={61–67} }
@book{Lewis_1981, title={Evaluation of copper-nickel alloys for OTEC applications}, url={http://dx.doi.org/10.2172/6255887}, DOI={10.2172/6255887}, institution={Office of Scientific and Technical Information (OSTI)}, author={Lewis, R. and LaQue, F.}, year={1981}, month=Apr }
/workspace/citations/local-copper-review/e8550274-e07d-4d00-9af7-1a2a7d7da

In [5]:
print({k:os.environ.get(k) for k in ['PROJECT_ID','TRAJECTORY_ID','WORKSPACE_PATH','PERSONA_ID']})
print([(t.done(),str(t.result())[:200] if t.done() else '') for t in paper_tasks])
for name in ['ravindran','banning']:
 text=(OUT/'sources'/f'{name}.txt').read_text()
 for token in (['61.0','22.5','Table 5','TABLE 5'] if name=='ravindran' else ['0.99','697','72','TABLE 7','TABLE 8']):
  p=text.find(token);print(name,token,p,'\n',text[max(0,p-700):p+1500])

{'PROJECT_ID': None, 'TRAJECTORY_ID': 'e8550274-e07d-4d00-9af7-1a2a7d7da0c0', 'WORKSPACE_PATH': '/workspace', 'PERSONA_ID': None}
[(True, '(\'10.56093/ft.v20i2.70952\', \'HTTPStatusError("Client error \\\'403 Forbidden\\\' for url \\\'https://prod.api.paperqa.app/api/paper\\\'\\\\nFor more information check: https://developer.mozilla.org/en-US/docs/W'), (True, "('10.2172/138275', {'schema_version': 'paperqa.parsing.v1', 'input': {'kind': 'doi', 'value': '10.2172/138275'}, 'resolved': {'paper_id': 'bb800cacfe935515', 'doi': '10.2172/138275', 'pmc_id': None, '"), (True, '(\'10.1016/j.desal.2007.08.007\', \'HTTPStatusError("Client error \\\'403 Forbidden\\\' for url \\\'https://prod.api.paperqa.app/api/paper\\\'\\\\nFor more information check: https://developer.mozilla.org/en-US/do'), (True, "('10.2172/6255887', {'schema_version': 'paperqa.parsing.v1', 'input': {'kind': 'doi', 'value': '10.2172/6255887'}, 'resolved': {'paper_id': 'd66d4818ee7997e3', 'doi': '10.2172/6255887', 'pmc_id': Non

In [6]:
print((OUT/'sources/ravindran.txt').read_text()[:1000])
papers=dict(t.result() for t in paper_tasks)
for d,p in papers.items():
 if isinstance(p,dict):
  print('\nPAPER',d,'keys',p.keys())
  print(str(p)[:1000])
print('\nBOFN immersion related lines:')
lines=Path('/workspace/bofn_alloy.py').read_text().splitlines()
for i,line in enumerate(lines):
 if any(s in line for s in ['REFERENCE =','def full','"beta"','"rho_m"','m_ref','M_REF','"m":','def node','def objective']): print(i+1,line)
print('Phys tables JSON keys:',json.loads(Path('/workspace/phys_tables.json').read_text()).keys())


--- PDF PAGE 1 ---
90
Copper-Nickel Alloys for Marine Aquacultural Engineering
K. RAVINDRAN
Central Institute of Fisheries Technology, Cochin - 682 029
Engineering developments, problems of marine corrosion, fouling and practical
innovative applications of copper-nickel alloys in comparison with other materials for
marine aquacultural engineering are presented .
Coefficient of drag and hydrodynamic
force acting on nylon and polyethylene net enclosures have shown a four-fold increase
of force in four months at the Cochin Harbour due to fouling . Corrosion behaviour of
copper-nickel and other copper base alloys are also presented.
Effects of copper addi-
tions to culture water are reviewed .
Recent advances in aquaculture presented
at the FAO Technical Conference at Kyoto,
Japan in 1976 were documented by Pillai &
Dill
(1979) .
World
production through
aquaculture amounts to 6.1 million t. of
which India contributes only 0.49 million t .
It is estimated that agricultural production
thro

In [7]:
from datetime import date
print('Today',date.today())
print((OUT/'sources/ravindran.txt').read_text()[-2500:])
for doi,name,tokens in [('10.2172/138275','gdowski',['Table 34','TABLE 34','12.1','localized']),('10.2172/6255887','lewis',['TABLE XIV','Table XIV','43.2','5.9','0.2-0.6'])]:
 p=papers[doi]
 text='\n'.join('\n--- PAGE '+str(i+1)+' ---\n'+str(pg.get('text','')) for i,pg in enumerate(p['pages']))
 (OUT/'sources'/f'{name}.txt').write_text(text)
 for tok in tokens:
  pos=text.lower().find(tok.lower()); print(name,tok,pos,'\n',text[max(0,pos-250):pos+3500])
print('\n'.join(f'{i+1}: {lines[i]}' for i in range(90,200)))

Today 2026-10-28
 the unfouled
net by current (N)
Force applied to the fouled net
by current (N)
C
Gallons per min . ft- 2 exposed
copper
C'
-
40C = Litres per min, m- 2 of
exposed copper
I am thankful to the International Nickel Com-
pany Inc ., New York for copper-nickel plates, to
N . V. Bekaert Societe Anonyme, Belgium for copper-
nickel and copper-nickel clad steel wire nettings and
to International Copper Research Inc ., New York
for INCRA Research Reports, to Shri R . Balasubra-
manyan for his interest in the work and to Dr .
C. C. Pandurango Rao, Director, Central Institute
of Fisheries Technology, Cochin for his encourage-
ment and permission to publish this paper .
References
Anon
(1975)
INCRA Project 251, Annual
Rep., INCRA, New York
Anon
(1979)
INCRA, 2, 1
Anon
(1981)
INCRA, 12,
1
Fcf
(m)
K. RAVINDRAN
Ansuini, F . J. & Huguenin, J . E. (1978)
Proc. 9th Annual Mtg. World Maricul-
ture Soc. p .
737
Efird, K. D. & Anderson, D. B .
(1975)
Mater . Perf. 14, 37
Hinde, J.
(1962)
M

In [8]:
REF={'C70600':np.array([10.,0.,1.4]),'C71500':np.array([30.,0.,.6])}
def score(f,c):return feature(c,f['cfg'])@f['beta']
def optimal_af(f):
 b=f['beta']; cand=[0.,9.5]
 if b[2]>1e-12: cand.append(np.clip(-5*b[1]/b[2],0,9.5))
 a=min(cand,key=lambda a:b[1]*a+b[2]*a*a/10)
 fe=f['cfg']['fe_sol'] if b[3]<-1e-9 else 0.
 return a,fe

def margins(c,mn=0):
 n,a,f=c
 return np.array([35-sum(c)-mn,1250-float(L([[n,a,f,mn]])[0]), (n-f if a>7+1e-9 else 100.),(2.3-f if a<1.5-1e-9 else 100.)])

def opt_floor(f,bar,mn=0):
 # Global unconstrained lower bound: exactly minimize separable Al and Fe terms,
 # then solve monotone Ni constraint; if its witness satisfies limits, it certifies the global floor.
 a,fe=optimal_af(f); target=score(f,REF[bar]); bn=f['beta'][0]
 def fun(n):return score(f,[n,a,fe])-target
 if fun(0)<=1e-10:n=0.
 elif bn<0 and fun(30)<=1e-10:n=brentq(fun,0,30,xtol=1e-12) if fun(30)<0 else 30.
 else:n=None
 if n is not None and min(margins([n,a,fe],mn))>=-1e-6:
  c=np.array([n,a,fe]);return dict(floor=n,witness=c.tolist(),mn=mn,liquidus=float(L([[*c,mn]])[0]),slack=float(target-score(f,c)),certificate='separable lower bound attained')
 # General fallback: optimize all continuous variables over each conditional branch.
 # Not needed for the base; report if used.
 sols=[]
 for amin,amax,fmax,couple in [(0,1.5,2.3,False),(1.5,7,4.5,False),(7,9.5,4.5,True)]:
  bounds=[(0,30),(amin,amax),(0,fmax)]
  def con(x):return np.r_[target-score(f,x),35-sum(x)-mn,1250-L([[*x,mn]])[0],x[0]-x[2] if couple else 100]
  for seed in range(10):
   rng=np.random.default_rng(seed);x0=np.array([rng.uniform(*bb) for bb in bounds])
   r=minimize(lambda x:x[0],x0,method='SLSQP',bounds=bounds,constraints=[{'type':'ineq','fun':con}],options={'ftol':1e-11,'maxiter':500})
   if r.success and min(con(r.x))>=-1e-7:sols.append(r)
 if not sols:return dict(floor=None,certificate='no feasible optimization result')
 r=min(sols,key=lambda s:s.fun);return dict(floor=r.fun,witness=r.x.tolist(),mn=mn,liquidus=float(L([[*r.x,mn]])[0]),slack=float(target-score(f,r.x)),certificate='multistart piecewise SLSQP')

specs=[('Base',{},df),('Linear Al',{'al_form':'lin'},df),('Signs free',{'signs':False},df),('Linear Al, signs free',{'al_form':'lin','signs':False},df),
 ('Ni scale 3',{'ni_scale':3},df),('Ni scale 8',{'ni_scale':8},df),('Ni scale 12',{'ni_scale':12},df),('Linear Ni',{'ni_form':'lin'},df),
 ('Fe cap 1.0',{'fe_sol':1},df),('Fe cap 2.5',{'fe_sol':2.5},df),('No ridge',{'ridge':0},df),('Ridge 0.5',{'ridge':.5},df),('Equal row weights',{},df.assign(weight=1.))]
for s in sorted(df.study.unique()):specs.append(('Drop '+s,{},df[df.study!=s]))
weak_new=['CohenWhitted1971','Wheatfall1967_123d','Wheatfall1967_surface']
specs += [('Drop hot brine + both Wheatfall blocks',{},df[~df.study.isin(weak_new)]),
 ('Drop unspecified bronze rows',{},df[~((df.study=='LewisLaQue1981_1y') & df.alloy.str.contains('generic'))]),
 ('Drop hot/deep/local + unspecified bronze + Zhao',{},df[~df.study.isin(weak_new+['Zhao2022']) & ~((df.study=='LewisLaQue1981_1y')&df.alloy.str.contains('generic'))]),
 ('Weight each study equally',{},df.assign(weight=df.weight/df.groupby('study').weight.transform('sum')))]
results=[];fits={}
for name,opts,dat in specs:
 f=own_fit(dat,**opts);fits[name]=f
 rr={'model':name,'n':f['n'],'rmse_ln':f['rmse'],'b_ni':f['beta'][0],'b_al':f['beta'][1],'b_al2':f['beta'][2],'b_fe':f['beta'][3]}
 for bar in REF:rr[bar]=opt_floor(f,bar)
 results.append(rr)
rt=pd.DataFrame([{**{k:v for k,v in r.items() if k not in REF},**{b:r[b]['floor'] for b in REF}} for r in results])
print(rt.to_string(index=False,float_format=lambda v:f'{v:.5f}'))
print('Base witnesses',results[0])
print('Certificates',pd.Series([r[b]['certificate'] for r in results for b in REF]).value_counts())
(OUT/'refits.json').write_text(json.dumps(results,indent=2))
rt.to_csv(OUT/'refits.csv',index=False)

                                          model  n  rmse_ln     b_ni     b_al   b_al2     b_fe   C70600   C71500
                                           Base 30  0.22670 -0.84092 -0.27994 0.38567 -0.00000  1.50950  2.49979
                                      Linear Al 30  0.32335 -0.72965  0.04038 0.00000 -0.00000 10.00000 30.00000
                                     Signs free 30  0.22341 -0.91232 -0.28648 0.38179  0.12880  0.40597  1.95621
                          Linear Al, signs free 30  0.32037 -0.81593  0.02877 0.00000  0.15324  4.60310 10.80690
                                     Ni scale 3 30  0.24888 -0.73831 -0.27240 0.38903 -0.00000  1.15024  1.31131
                                     Ni scale 8 30  0.20430 -0.93489 -0.28251 0.38267 -0.00000  1.35467  4.34073
                                    Ni scale 12 30  0.19269 -0.99539 -0.27718 0.37969 -0.02999  0.66856  5.78689
                                      Linear Ni 30  0.18361 -0.28362 -0.25864 0.37837 -0.20780  

In [9]:
import importlib.util, sys
print({k:importlib.util.find_spec(k) is not None for k in ['torch','botorch','gpytorch']})
print('Gdowski relevant page:')
t=(OUT/'sources/gdowski.txt').read_text();p=t.find('Table 34.');print(t[p:p+12500])
print('Lewis relevant table:')
t=(OUT/'sources/lewis.txt').read_text()
for token in ['TABLE XIV','Table XIV','43.2']:
 p=t.find(token);print(token,p);print(t[max(0,p-250):p+8500])

{'torch': True, 'botorch': False, 'gpytorch': False}
Gdowski relevant page:
Table 34.  General  corrosion  experiments   in seawater  (Wheatfail   1967).  Exposure  conditions  are shown  in
Table 35.

                      Corrosion  rate (todd a)
                         Exposure  No.                        Surface  attack                 Crevice  attack
  Alloy         1         2         3         4          (outside  crevice  area)       (maximum   pit depth)

Type 304      5.91 b      _.51 b      5.31 b      3.15 b    No visible  pits in No.  1.         Inside  crevice:
                                                          Severe  pitting and edge        No. 1 = 790 }J.m,
                                                                attack  at scattered  areas       Nos.  2, 3 = 3200 pm
                                                                      in Nos. 2 and  3. Attack          (perforated),
                                                           along  ali ed

In [10]:
import subprocess, time
p=subprocess.run([sys.executable,'-m','pip','-q','install','botorch'],capture_output=True,text=True,timeout=240)
print('Install exit',p.returncode,p.stderr[-1800:])
# Verify table XIV alone, which is on source PDF page 53 or nearby.
p=papers['10.2172/6255887']
for i,pg in enumerate(p['pages']):
 txt=pg.get('text','')
 if 'TABLE XIV' in txt or ('43.2' in txt and 'bronze' in txt.lower()):print('Lewis PDF page',i+1,txt[:11000])
# Find Al-Fozan author's publicly hosted source.
ss=await web_search('"Effect of seawater level" "pdf" Malik Al-Fozan',num_results=8)
for s in ss:print(s['title'],s['url'])
(OUT/'sources/alfozan_search.json').write_text(json.dumps(ss,indent=2))

Install exit 0 
Lewis PDF page 55                          TABLE XIV

       Galvanic Couple Data For C70600  and C71500
    With Other Materials in 0.6 m/s Flowing Seawater

         One Year Exposures - Equal Area Couples
                    (Inco Unpublished Data)

Uncoupled                                   Corrosion Rate
                                             pm/y  (mpy)


  C70600                                         30.5  (1.2)
  C71500                                        20.3  (0.8)
  Al Bronze                                    43.2  (1.7)
  Ni Al Bronze                                150.0  (5.9)
  Carbon Steel                                330.0  (13.0)
  Titanium                                     <2.5  (<0.1)



Couples


  C70600                                        25.4  (1.0)
  Al Bronze                                     43.2  (1.7)

  C70600      •                                    152.4  (6.0)
  Ni Al Bronze                                 81.3  (3.

Seawater Levels Impact on Alloy Corrosion | PDF | Science & Mathematics | Technology & Engineering https://www.scribd.com/doc/315238285/Effect-of-Seawater-Level-on-Corrosion-Behavior-of-Different
Effect of seawater level on corrosion behavior of different ... https://www.researchgate.net/publication/238095001_Effect_of_seawater_level_on_corrosion_behavior_of_different_alloys
Corrosion Behavior of Stainless Steel in Seawater in the ... https://www.mdpi.com/2076-3417/13/7/4366
Influence of water salinity on corrosion risk—the case of ... - PMC https://pmc.ncbi.nlm.nih.gov/articles/PMC4064120/
Comparative Studying of Marine Parameters’ Effect, via Quantitative Method https://scialert.net/fulltext/?doi=tasr.2011.65.72
Driving forces in the corrosion dynamics of stainless steels ... https://iopscience.iop.org/article/10.1088/2053-1591/ae50d6/pdf
(PDF) Assessment of Corrosion in Offshore Structures through Inspection Reports and Experiments https://www.academia.edu/78465272/Assessment%5Fof%5

57639

In [11]:
import torch
# Uploaded layout is flat, whereas the truth hard-codes phys/. I override paths in memory only.
import bofn_alloy as BA
import truth_phys as TP
TP.GRID_CSV=Path('/workspace/calphad_grid.csv');TP.FIT_JSON=Path('/workspace/corrosion_fit.json')
TP._GRID=None;TP._FIT=None
truth=TP.PhysTruth(m_ref=0.)
print('Fit identity',TP.fit_id())
for name in BA.REFERENCE:
 print(name,'wt',TP.wt(torch.tensor(BA.REFERENCE[name])).tolist(),'best sobol ref',TP.reference_m(name)[:2])
refs={b:TP.reference_m(name) for b,name in zip(REF,['C70600 (90-10 Cu-Ni)','C71500 (70-30 Cu-Ni)'])}
pt=json.loads(Path('/workspace/phys_tables.json').read_text())
for b,key in [('C70600','9010'),('C71500','7030')]:
 x=torch.tensor(pt['immersion'][key]['witness']['x']);out=truth.full(x)
 print('Uploaded full-truth floor witness',b,'Ni',30*float(x[0]),'m',float(out['m']),'bar',refs[b][0],'feasible',bool(TP.feasible(x)),'rho',float(out['rho_m']),'beta',float(out['beta']))
# Independent fit function vs shipped fitter: now execute only fitter (not its expensive grid-floor main).
import fit_corrosion as FC
shipped=FC.fit(FC.load())
print('Independent vs executed shipped fitter max abs coefficient error',max(abs(base['beta'][i]-shipped['coef'][k]) for i,k in enumerate(['ni','al','al2','fe'])))
# Source scraping attempts run asynchronously while numerical review proceeds.
from e14c.web import fetch_parsed_page
async def getpage(name,url):
 try:
  txt=await fetch_parsed_page(url);(OUT/'sources'/f'{name}.md').write_text(txt);return name,len(txt),txt[:100]
 except Exception as e:return name,repr(e)
scrape_tasks=[asyncio.create_task(getpage('alfozan_scribd','https://www.scribd.com/doc/315238285/Effect-of-Seawater-Level-on-Corrosion-Behavior-of-Different')),asyncio.create_task(getpage('ravindran_meta','https://epubs.icar.org.in/index.php/FT/article/view/70952'))]

Fit identity e8d940d600


C71500 (70-30 Cu-Ni) wt [30.0, 0.0, 0.6749999999999999, 0.8] best sobol ref (0.8420666180680609, 0.9735553935602034)
C70600 (90-10 Cu-Ni) wt [9.99, 0.0, 1.395, 0.8] best sobol ref (0.8104866235013652, 0.9802384084392695)
C95800 (NAB) wt [4.5, 9.025, 4.05, 1.2] best sobol ref (0.27350543472437017, 0.9660663574274118)


Uploaded full-truth floor witness C70600 Ni 2.24639937330599 m 0.8104868961706859 bar 0.8104866235013652 feasible True rho 0.9867440062321694 beta 0.11512544180670455
Uploaded full-truth floor witness C71500 Ni 3.29276658405919 m 0.842071669972141 bar 0.8420666180680609 feasible True rho 0.9846597442920095 beta 0.10716163789363002
Independent vs executed shipped fitter max abs coefficient error 2.8341692674681178e-11


In [12]:
# Add both other matched Wheatfall exposures to test selection; retain independent intercepts.
more=[]
for name,rates in [('Wheatfall1967_751d',(3.78,2.75)),('Wheatfall1967_1064d',(2.65,3.08))]:
 for ni,rate in zip([0.,30.],rates):
  row=df[(df.study=='Wheatfall1967_123d')&(df.ni==ni)].iloc[0].copy();row['study']=name;row['rate']=rate;more.append(row)
df_wheat=pd.concat([df,pd.DataFrame(more)],ignore_index=True)
prune=['Tan2024','Qian2025','Zhao2022','LewisLaQue1981_quiet',*weak_new]
dclean=df[~df.study.isin(prune)&~((df.study=='LewisLaQue1981_1y')&df.alloy.str.contains('generic'))]
extra_specs=[('Add omitted 751d + 1064d Wheatfall blocks',{},df_wheat),('Drop all Wheatfall exposures',{},df[~df.study.str.startswith('Wheatfall')]),('Exposure-rate only, conservative exclusions (9 rows)',{},dclean),('Same exclusions, no 10d Yang (7 rows)',{},dclean[dclean.study!='Yang2018'])]
results=[r for r in results if r['model'] not in {x[0] for x in extra_specs}]
for name,opts,dat in extra_specs:
 f=own_fit(dat,**opts);fits[name]=f
 rr={'model':name,'n':f['n'],'rmse_ln':f['rmse'],'b_ni':f['beta'][0],'b_al':f['beta'][1],'b_al2':f['beta'][2],'b_fe':f['beta'][3],'rank':int(f['rank'])}
 for bar in REF:rr[bar]=opt_floor(f,bar)
 results.append(rr)
 print(name,'n',len(dat),'rank',f['rank'],'coef',f['beta'],'floors',[rr[b]['floor'] for b in REF])
(OUT/'refits.json').write_text(json.dumps(results,indent=2,default=lambda o:o.item() if isinstance(o,np.generic) else o))
print('All source scraping attempts completed:',[t.result() for t in scrape_tasks])

Add omitted 751d + 1064d Wheatfall blocks n 34 rank 4 coef [-6.64419005e-01 -2.57028405e-01  3.73393680e-01 -3.85015735e-18] floors [1.109110914704992, 2.0158292993910747]
Drop all Wheatfall exposures n 26 rank 4 coef [-8.11744883e-01 -2.76150974e-01  3.83640987e-01 -3.18600013e-19] floors [1.4549133440987803, 2.433331600949502]
Exposure-rate only, conservative exclusions (9 rows) n 9 rank 3 coef [-1.05229185e+00 -1.90645041e-01 -5.71935124e-02 -4.61661626e-17] floors [0.0, 0.0]
Same exclusions, no 10d Yang (7 rows) n 7 rank 2 coef [-1.05283551e+00  0.00000000e+00  0.00000000e+00 -1.71528705e-17] floors [10.000000000000107, 30.0]
All source scraping attempts completed: [('alfozan_scribd', "HTTPStatusError('Failed to scrape https://www.scribd.com/doc/315238285/Effect-of-Seawater-Level-on-Corrosion-Behavior-of-Different: HTTP 503 with description None')"), ('ravindran_meta', "HTTPStatusError('Failed to scrape https://epubs.icar.org.in/index.php/FT/article/view/70952: HTTP 503 with descri

In [13]:
torch.set_num_threads(2)
# Evaluate the actual supplied truth. Finite differences avoid an in-place operation in transform()
# that prevents torch autograd through rho_u; I do not alter the supplied physical functions.
class EvalTruth:
 def __init__(self,bar,beta_mult=1.,porosity_mult=1.):self.bar=bar;self.beta_mult=beta_mult;self.porosity_mult=porosity_mult
 def fun(self,x):
  t=torch.tensor(x);o=truth.full(t)
  m=(TP.M_LEVEL-TP.SLOPE*TP.ln_rate(t[:4])-.60*t[1]*o['beta']*self.beta_mult-.30*(1-o['rho_m'])*self.porosity_mult)
  return np.array([float(m)-self.bar, (35.-float((t[:4]*TP.WT_MAX).sum()))/35.,(1250.-float(TP.field(t[:4],'liquidus_C')))/100.])

def full_floor(bar,starts=6,beta_mult=1.,porosity_mult=1.,seed=12):
 ev=EvalTruth(bar,beta_mult,porosity_mult);sols=[]
 # The relevant low-floor branch has 1.5 <= Al <= 7; these local results are not global certificates.
 bounds=[(0,1),(1.5/9.5,7/9.5),(0,1),(0,1),(0,1),(0,1),(0,1)]
 rng=np.random.default_rng(seed)
 x0s=[np.array(pt['immersion'][b]['witness']['x']) for b in ['9010','7030']]
 for j in range(starts-2):
  x=x0s[j%2].copy();x[:4]+=rng.normal(0,.08,4);x[4:]+=rng.normal(0,.04,3)
  x=np.clip(x,np.array(bounds)[:,0],np.array(bounds)[:,1]);x0s.append(x)
 for x0 in x0s:
  r=minimize(lambda x:x[0]*30,x0,jac=lambda x:np.array([30.,0,0,0,0,0,0]),method='SLSQP',bounds=bounds,constraints=[{'type':'ineq','fun':ev.fun}],options={'maxiter':200,'ftol':1e-10})
  if r.success and ev.fun(r.x).min()>-1e-7:sols.append(r)
 if not sols:return {'failed':True}
 r=min(sols,key=lambda r:r.fun);xx=torch.tensor(r.x);o=truth.full(xx)
 return {'ni':r.fun,'x':r.x.tolist(),'wt':TP.wt(xx).tolist(),'m_original':float(o['m']),'rho':float(o['rho_m']),'beta':float(o['beta']),'constraint_margin':ev.fun(r.x).tolist(),'successes':len(sols),'local_ni_values':[float(s.fun) for s in sols]}
fullresults={}
t0=time.time()
for b in REF:
 fullresults[b]=full_floor(refs[b][0],starts=6)
 print(b,fullresults[b])
print('Seconds',time.time()-t0)
(OUT/'full_truth_optimizations.json').write_text(json.dumps(fullresults,indent=2))

C70600 {'ni': np.float64(2.2048496640200916), 'x': [0.07349498880066972, 0.30601597499259175, 0.8759000904245616, 7.093870937129939e-16, 0.4262649105313738, 0.6286440738825333, 0.5279464726785607], 'wt': [2.2048496640200916, 2.9071517624296215, 3.9415504069105274, 2.8375483748519754e-15], 'm_original': 0.8104866234847239, 'rho': 0.9852306451201104, 'beta': 0.10757501325496654, 'constraint_margin': [-1.6641354960711396e-11, 0.7413270904754216, 1.2350159992990506], 'successes': 6, 'local_ni_values': [2.204849664510301, 2.204849664520078, 2.2048496645219826, 2.204849664489414, 2.2048496640200916, 2.204849664516618]}


C71500 {'ni': np.float64(3.2636371076706485), 'x': [0.10878790358902161, 0.30664508756779374, 0.8853512929413565, 0.0, 0.42328650774356424, 0.6246141245926171, 0.5258338189348591], 'wt': [3.2636371076706485, 2.9131283318940406, 3.9840808182361043, 0.0], 'm_original': 0.8420666180549575, 'rho': 0.9863191940141459, 'beta': 0.10638989942426005, 'constraint_margin': [-1.310340724813841e-11, 0.7096901069199774, 1.2312734874190279], 'successes': 6, 'local_ni_values': [3.2636371076706485, 3.263637107772881, 3.263637108159266, 3.263637108159859, 3.263637108158286, 3.2636371081593127]}
Seconds 15.796299457550049


1563

In [14]:
# Additional accessible full-text routes for Al-Fozan: record limitations if none succeeds.
search_qs=['"Al-Fozan" "0.145" "0.067"','"Effect of seawater level" filetype:pdf','"Effect of seawater level" swcc']
al_search=await asyncio.gather(*(web_search(q,num_results=5) for q in search_qs))
for q,ss in zip(search_qs,al_search):
 print(q)
 for s in ss:print(s['title'],s['url'],s.get('snippet','')[:250])
# Use raw requests to reach article metadata; the PDF itself is already downloaded and read.
u='https://epubs.icar.org.in/index.php/FT/article/view/70952'
r=requests.get(u,timeout=30)
from bs4 import BeautifulSoup
soup=BeautifulSoup(r.text,'html.parser')
meta={m.get('name'):m.get('content') for m in soup.find_all('meta') if 'citation_' in (m.get('name') or '')}
print('Ravindran publisher metadata',meta)
(OUT/'sources/ravindran_metadata.json').write_text(json.dumps(meta,indent=2))
# Canonical references used by code and fitting are not precisely equal.
for b,n in zip(REF,['C70600 (90-10 Cu-Ni)','C71500 (70-30 Cu-Ni)']):
 w=TP.wt(torch.tensor(BA.REFERENCE[n])).numpy()
 print(b,'fit reference',REF[b],'truth reference',w,'pure dense m',.85-.30*(score(base,w[:3])-score(base,TP.wt(torch.tensor(BA.REFERENCE['C71500 (70-30 Cu-Ni)'])).numpy()[:3])))
# True dense comparator values, as distinct from deposited porosity penalties.
dense_bars={b:.85-.30*(score(base,TP.wt(torch.tensor(BA.REFERENCE[n])).numpy()[:3])-score(base,TP.wt(torch.tensor(BA.REFERENCE['C71500 (70-30 Cu-Ni)'])).numpy()[:3])) for b,n in zip(REF,['C70600 (90-10 Cu-Ni)','C71500 (70-30 Cu-Ni)'])}
print('Ideal dense reference bars',dense_bars)
for b in REF:
 fullresults[b+'_ideal_dense_ref']=full_floor(dense_bars[b],starts=3)
 print(b,'floor against genuinely dense reference',fullresults[b+'_ideal_dense_ref']['ni'])
(OUT/'full_truth_optimizations.json').write_text(json.dumps(fullresults,indent=2))

"Al-Fozan" "0.145" "0.067"
Al Fozan Holding https://alfozan.com.sa/ Al Fozan Holding

### Our Businesses

Al Fozan Holding Company has grown to a progressive and dynamic conglomerate as a result of its keen focus on diversifying its portfolio of owned and managed assets. Our expertise and investment extend to a broad
MMSZ5227B | Diodes Inc. https://thepartsdirect.com/part/mmsz5227b~diodesinc MMSZ5227B | Diodes Inc.

# MMSZ5227B

 Diodes Inc.

 Also known as: Anachip, ICT, Integrated Circuit Technology, Pericom Semiconductor, Vishay/Lite-On Semiconductors, Zetex Semi

 Description: DIODES INC. MMSZ5227B DIODE ZENER 3.6V 0.5W SOD123

 Avg.
Al Fozan Holding | الفوزان القابضة https://linkedin.com/company/alfozanholding # Al Fozan Holding | الفوزان القابضة

Al Fozan Holding | الفوزان القابضة is a Investment Management company. Al Fozan Holding is a leading family business in Saudi Arabia with a diversified investment portfolio in trading, real estate, manufacturing,
Al Fozan Holding - The T

Ravindran publisher metadata {'citation_journal_title': 'Fishery Technology', 'citation_journal_abbrev': 'Fish. Technol.', 'citation_issn': '2582-2632', 'citation_author': 'K RAVINDRAN', 'citation_title': 'COPPER - NICKEL ALLOYS FOR MARINE AQUA CULTURAL ENGINEERING', 'citation_language': 'en', 'citation_date': '1983', 'citation_volume': '20', 'citation_issue': '2', 'citation_doi': '10.56093/ft.v20i2.70952', 'citation_abstract_html_url': 'https://epubs.icar.org.in/index.php/FT/article/view/70952', 'citation_abstract': 'Engineering developments, problems of marine corrosion, fouling and practicalinnovative applications of copper-nickel alloys in comparison with other materials for marine aquacultural engineering are presented. Coefficient of drag and hydrodynamic force acting on nylon and polyethylene net enclosures have shown a four-fold increase of force in four months at the Cochin Harbour due to fouling. Corrosion behaviour of copper-nickel and other copper base alloys are also prese

C70600 floor against genuinely dense reference 2.387421361700049


C71500 floor against genuinely dense reference 3.568544860880492


3031

In [15]:
# Direct HTML can expose the document text even if browser scraping is unavailable.
al_urls=['https://www.scribd.com/doc/315238285/Effect-of-Seawater-Level-on-Corrosion-Behavior-of-Different','https://www.sciencedirect.com/science/article/pii/S0011916408002048','https://api.elsevier.com/content/article/doi/10.1016/j.desal.2007.08.007?httpAccept=text/xml']
for i,u in enumerate(al_urls):
 try:
  r=requests.get(u,timeout=30);text=BeautifulSoup(r.text,'html.parser').get_text(' ',strip=True)
  (OUT/'sources'/f'alfozan_attempt_{i}.txt').write_text(text)
  print(i,r.status_code,len(r.content),len(text),text[:120])
  for tok in ['0.145','0.067','88.18','198']:
   pos=text.find(tok)
   if pos>=0:print(tok,text[max(0,pos-300):pos+800])
 except Exception as e:print(type(e).__name__,str(e)[:180])
# Correct the no-DOI citation using the actual PDF and publisher metadata, not guessed pages.
print(save_citation('''@article{ravindran1983marine,
 author={Ravindran, K.}, title={Copper-Nickel Alloys for Marine Aquacultural Engineering},
 journal={Fishery Technology}, volume={20}, number={2}, pages={90--96}, year={1983},
 doi={10.56093/ft.v20i2.70952},
 url={https://epubs.icar.org.in/index.php/FT/article/download/70952/60283/468475}
}'''))
# cite_doi cannot resolve the publisher-listed DOI (404), so save_citation records verified metadata.
print(save_citation('''@online{banning1973hotbrines,
 author={Banning, L. H. and Oden, L. L.},
 title={Corrosion Resistance of Metals in Hot Brines: A Literature Review},
 date={1973}, institution={U.S. Bureau of Mines}, number={Information Circular 8601},
 url={https://stacks.cdc.gov/view/cdc/235753/cdc_235753_DS1.pdf}, urldate={2026-10-28}
}'''))
print(save_citation('''@online{gdowski1991degradation,
 author={Gdowski, G. E.}, title={Survey of Degradation Modes of Four Nickel-Chromium-Molybdenum Alloys},
 date={1991}, institution={Lawrence Livermore National Laboratory}, doi={10.2172/138275},
 url={https://doi.org/10.2172/138275}, urldate={2026-10-28}
}'''))

0 200 3038 315 Client Challenge JavaScript is disabled in your browser. Please enable JavaScript to proceed. A required part of this si
1 403 832804 641 ScienceDirect ScienceDirect Help There was a problem providing the content you requested Please contact our support team


2 200 1806 306 https://api.elsevier.com/content/article/pii/S0011916408002324 doi:10.1016/j.desal.2007.08.007 1-s2.0-S0011916408002324 
/workspace/citations/local-copper-review/e8550274-e07d-4d00-9af7-1a2a7d7da0c0.bib
/workspace/citations/local-copper-review/e8550274-e07d-4d00-9af7-1a2a7d7da0c0.bib
/workspace/citations/local-copper-review/e8550274-e07d-4d00-9af7-1a2a7d7da0c0.bib


In [16]:
r=requests.get('https://api.elsevier.com/content/article/doi/10.1016/j.desal.2007.08.007?httpAccept=text/xml',timeout=30)
print(r.text)
# Save a compact numerical audit of witnesses and all limits, with both reference conventions.
rows=[]
for rr in results:
 for b in REF:
  o=rr[b]; c=o.get('witness')
  if c is None:continue
  ma=margins(c)
  rows.append(dict(model=rr['model'],bar=b,ni=c[0],al=c[1],fe=c[2],mn=o['mn'],cu=100-sum(c)-o['mn'],liquidus_C=o['liquidus'],ln_rate_slack=o['slack'],min_feasibility_margin=ma.min(),certificate=o['certificate']))
witness_df=pd.DataFrame(rows);witness_df.to_csv(OUT/'optimized_witnesses.csv',index=False)
rt=pd.DataFrame([{**{k:v for k,v in r.items() if k not in REF},**{b:r[b]['floor'] for b in REF}} for r in results]);rt.to_csv(OUT/'refits.csv',index=False)
print('Minimum feasibility margin over all film-model witnesses',witness_df.min_feasibility_margin.min())
print('Max absolute active rate slack',witness_df[witness_df.ni>1e-7].ln_rate_slack.abs().max())
# Algebraic diagnostic: base Al minimum and compensation relative to copper.
a0=optimal_af(base)[0];help_al=base['beta'][1]*a0+base['beta'][2]*a0*a0/10
print('Base optimum Al',a0,'ln-rate gain',help_al,'rate multiplier',np.exp(help_al))
# Sensitivity to the two handwritten immersion penalties, keeping the observed bar fixed.
for label,bmult,pmult in [('No beta penalty',0,1),('Double beta penalty',2,1),('No porosity penalty',1,0),('Double porosity penalty',1,2)]:
 rr={b:full_floor(refs[b][0],starts=3,beta_mult=bmult,porosity_mult=pmult) for b in REF}
 fullresults[label]=rr
 print(label, {b:v.get('ni') for b,v in rr.items()})
(OUT/'full_truth_optimizations.json').write_text(json.dumps(fullresults,indent=2))

<full-text-retrieval-response xmlns="http://www.elsevier.com/xml/svapi/article/dtd" xmlns:bk="http://www.elsevier.com/xml/bk/dtd" xmlns:cals="http://www.elsevier.com/xml/common/cals/dtd" xmlns:ce="http://www.elsevier.com/xml/common/dtd" xmlns:ja="http://www.elsevier.com/xml/ja/dtd" xmlns:mml="http://www.w3.org/1998/Math/MathML" xmlns:sa="http://www.elsevier.com/xml/common/struct-aff/dtd" xmlns:sb="http://www.elsevier.com/xml/common/struct-bib/dtd" xmlns:tb="http://www.elsevier.com/xml/common/table/dtd" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns:xocs="http://www.elsevier.com/xml/xocs/dtd" xmlns:dc="http://purl.org/dc/elements/1.1/" xmlns:prism="http://prismstandard.org/namespaces/basic/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"><coredata><prism:url>https://api.elsevier.com/content/article/pii/S0011916408002324</prism:url><dc:identifier>doi:10.1016/j.desal.2007.08.007</dc:identifier><eid>1-s2.0-S0011916408002324</eid><prism:doi>10.1016/j.desal.2007.08.007</prism:do

No beta penalty {'C70600': np.float64(1.4420037453769516), 'C71500': np.float64(2.3481177394230914)}


Double beta penalty {'C70600': np.float64(2.7120301116726524), 'C71500': np.float64(3.892109271885546)}


No porosity penalty {'C70600': np.float64(1.3578745236000616), 'C71500': np.float64(2.2548236706862497)}


Double porosity penalty {'C70600': np.float64(2.3229209853626664), 'C71500': np.float64(3.3973041468368224)}


9389

In [17]:
# Make the source audit machine-readable, using CSV line numbers (including header).
audit=[]
def add_audit(lines_,source,status,issue):
 for ln in lines_:
  rr=df.iloc[ln-2]
  audit.append(dict(csv_line=ln,study=rr.study,alloy=rr.alloy,rate=rr.rate,unit=rr.unit,source=source,verification=status,qualification=issue))
add_audit([18,19,20],'Ravindran 1983 Table 5 (PDF p.5)','Primary PDF inspected; rates verified','Same 120d natural seawater panels, tidal flushing 0–0.8 m/s; not 30d quiet immersion; method unstated. Ni nominal; Cu–Ni Fe/Mn not assayed or supplied in Table 5; CSV Fe/Mn assumed.')
add_audit([21,22],'Wheatfall 1967 via Gdowski 1991 Tables 34–35 (PDF p.45)','Compilation inspected; rates verified','Matched 123d mass losses, but 5640ft / 2.78 C / oxygen 1.19 ppm / pressure 2500 psi; not ambient surface seawater. Cu–Ni Fe/Mn assumed.')
add_audit([23,24],'Wheatfall 1967 via Gdowski 1991 Tables 34–35 (PDF p.45)','Compilation inspected; rates and localized-loss flag verified','Matched 386d surface exposure; Cu 4.50 mdd explicitly largely localized, not uniform film penetration. Cu–Ni Fe/Mn assumed. Two other matched exposure pairs omitted from CSV.')
add_audit([25,26,27,28,29],'Cohen–Whitted 1971 via Banning–Oden 1973 Tables 7–8 (PDF pp.18–19)','Compilation inspected; same-column rates verified','Same 697d / 4 ft/s column, but 250 F (~121 C), treated oxygen-poor brine (72 ppb makeup O2), 83% seawater salinity. Nominal specification limits, not assays; C613 Fe 2.5 unsupported and 0.2–0.5% Sn omitted.')
add_audit([30,31],'Al-Fozan & Malik 2008 Tables 1,3 per copper-anchor.md','Checked against uploaded anchor only; full source inaccessible','Rates/compositions agree with anchor; 198d vs 192d, not exact same-duration comparison. Measured Ni/Fe/Mn per anchor. DOI metadata verified, not table values.')
pd.DataFrame(audit).to_csv(OUT/'new_rows_audit.csv',index=False)
print('Audited rows',len(audit))
# Validate source page count for correct Ravindran citation pages.
text=(OUT/'sources/ravindran.txt').read_text()
for p in text.split('--- PDF PAGE ')[1:]:print(p.split('---')[0],p.split('---')[1][:90].replace('\n',' '))
print('Unique fit variants',len(results))
# Reproducible self-contained analysis script, with notebooks' executed independent definitions.
# Export only successful cells needed for fitting and optimization, not transient web attempts.
hist=get_ipython().history_manager.input_hist_raw
print('History entries',len(hist))
# Primary-source raw responses saved by fetch_paper are already local JSON.
print('Output files',[(x.name,x.stat().st_size) for x in OUT.iterdir() if x.is_file()])

Audited rows 14
1   90 Copper-Nickel Alloys for Marine Aquacultural Engineering K. RAVINDRAN Central Institut
2   CUPRONICKEL FOR MARICULTURE Table 1 . Typical properties and applications of wrought cupr
3   92 K . RAVINDRAN Table 2. Weight increase of netting materials due to biofouling in sea w
4   CUPRONICKEL FOR MARICULTURE During the course of investigations under INCRA sponsored pro
5   94 K. RAVINDRAN Table 4. Coefficient of drag andforce applied to nylon and polyethylene n
6   CUPRONICKEL FOR MARICULTURE 1 2 3 3 4 0 Time , Days x 10 Fig . 1 . Chronogravimetric curv
7   96 could be seen from the graph that a value of C= 10 can be allowed while using copper- 
Unique fit variants 32
History entries 23
Output files [('full_truth_optimizations.json', 9389), ('independent_base.npz', 294), ('refits.json', 25761), ('optimized_witnesses.csv', 10857), ('refits.csv', 5187), ('new_rows_audit.csv', 4938)]


In [18]:
# A clear interior counterexample under the actual supplied truth.
x=np.array(fullresults['C70600']['x']);x[0]=5/30
v=truth.full(torch.tensor(x))
print('Ni5 witness wt',TP.wt(torch.tensor(x)).tolist(),'m',float(v['m']),'bars',[refs[b][0] for b in REF],'feasible',bool(TP.feasible(torch.tensor(x))))
z=torch.tensor(pt['optimum']['x']);o=truth.full(z)
print('Uploaded optimum spot-check m',float(o['m']),'J',float(BA.objective_from(o['U'],o['m'],z,refs['C70600'][0])))
import ast
hist=get_ipython().history_manager.input_hist_raw
names={'feature','own_fit','score','optimal_af','margins','opt_floor'};blocks={}
for cell in hist:
 try:tree=ast.parse(cell)
 except SyntaxError:continue
 for node in tree.body:
  if isinstance(node,ast.FunctionDef) and node.name in names:blocks[node.name]=ast.get_source_segment(cell,node)
header='''"""Independent film-model refit and continuous nickel-floor optimization.
Run: python outputs/reproduce_refits.py [input_directory]
No supplied fitting module is imported. Outputs go to outputs/reproduction/.
"""
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from scipy.optimize import lsq_linear, minimize, brentq
from scipy.interpolate import RegularGridInterpolator
ROOT=Path(sys.argv[1]) if len(sys.argv)>1 else Path(__file__).resolve().parent.parent
OUT=ROOT/'outputs'/'reproduction'; OUT.mkdir(parents=True,exist_ok=True)
df=pd.read_csv(ROOT/'corrosion_data.csv')
BASE=dict(ni_scale=5.,ni_form='sat',al_form='quad',fe_sol=1.5,ridge=.05,signs=True)
REF={'C70600':np.array([10.,0.,1.4]),'C71500':np.array([30.,0.,.6])}
grid=pd.read_csv(ROOT/'calphad_grid.csv')
assert len(grid)==864 and (grid.ok==1).all()
axes=[np.sort(grid[c].unique()) for c in ['Ni_wt','Al_wt','Fe_wt','Mn_wt']]
L=RegularGridInterpolator(axes,grid.liquidus_C.fillna(1550).to_numpy().reshape(8,6,6,3),bounds_error=False,fill_value=None)
'''
script=header+'\n\n'+'\n\n'.join(blocks[n] for n in ['feature','own_fit','score','optimal_af','margins','opt_floor'])+'\n\n'
cell7=next(c for c in hist if c.startswith("REF={'C70600'") and "specs=[('Base'" in c)
script+=cell7[cell7.index("specs=[('Base'"):cell7.index('results=[];fits={}')]
cell11=next(c for c in reversed(hist) if c.startswith('# Add both other matched Wheatfall'))
script+=cell11[cell11.index('more=[]'):cell11.index('results=[r for r in results')]
script+='''\nspecs+=extra_specs
results=[]
for name,opts,dat in specs:
 f=own_fit(dat,**opts)
 rr=dict(model=name,n=f['n'],rank=int(f['rank']),rmse_ln=float(f['rmse']),coef=f['beta'].tolist())
 for bar in REF:rr[bar]=opt_floor(f,bar)
 results.append(rr)
(OUT/'refits.json').write_text(json.dumps(results,indent=2))
print('Computed',len(results),'optimized scenarios')
for r in results:print(r['model'],r['C70600']['floor'],r['C71500']['floor'])
'''
(OUT/'reproduce_refits.py').write_text(script)
p=subprocess.run([sys.executable,str(OUT/'reproduce_refits.py')],capture_output=True,text=True,timeout=180)
print('Reproduction exit',p.returncode,p.stdout[:1800],p.stderr[-1000:])
if p.returncode==0:
 rep=json.loads((OUT/'reproduction/refits.json').read_text())
 print('Reproduction max floor deviation',max(abs(a[b]['floor']-bb[b]['floor']) for a,bb in zip(results,rep) for b in REF))

Ni5 witness wt [5.0, 2.9071517624296215, 3.9415504069105274, 2.8375483748519754e-15] m 0.88113370115366 bars [0.8104866235013652, 0.8420666180680609] feasible True
Uploaded optimum spot-check m 0.9248101647375967 J 0.42931758870978115


Reproduction exit 0 Computed 32 optimized scenarios
Base 1.5094989098174514 2.499794137983035
Linear Al 10.000000000000101 30.0
Signs free 0.4059734523034528 1.9562108030456102
Linear Al, signs free 4.603097316884794 10.806902547025667
Ni scale 3 1.1502352096530644 1.3113148681349724
Ni scale 8 1.3546712379618964 4.340733396952579
Ni scale 12 0.6685566349795947 5.786886072279977
Linear Ni 0.0 7.822196574736614
Fe cap 1.0 1.5094989098174536 2.4997941379830366
Fe cap 2.5 1.5094989098174514 2.499794137983035
No ridge 1.4962335237466025 2.4836278690532514
Ridge 0.5 1.658882216441559 2.6825020171866263
Equal row weights 0.9178474956076833 1.7873941994260325
Drop AlFozan2008 1.4585694664916742 2.4377783168789633
Drop CohenWhitted1971 2.8957064190554664 4.2487378749108915
Drop LewisLaQue1981_1y 1.108891635430011 2.015566422595956
Drop LewisLaQue1981_quiet 1.517693519042364 2.5097854427077757
Drop Qian2025 1.4785383681999136 2.462077692290004
Drop Ravindran1983 1.2580428807467157 2.19490911043

In [19]:
# The two bars share the same global Al/Fe optimum in all computed film-model fits.
for r in results:
 a=r['C70600']['witness'];b=r['C71500']['witness']
 if not np.allclose(a[1:],b[1:]):print('Different AF:',r['model'],a,b)
print(witness_df[['model','bar','ni','al','fe']].tail(10).to_string(index=False))
print('Base 90/10 dense film rate / 70/30',np.exp(score(base,REF['C70600'])-score(base,REF['C71500'])))
# Export exact environment versions for reproducibility.
import scipy, importlib.metadata as md
versions={k:md.version(k) for k in ['numpy','pandas','scipy','torch','botorch']}
(OUT/'environment.json').write_text(json.dumps(versions,indent=2))
print(versions)
# Persist only a requested project workflow convention, not findings.
prompts=Path('/workspace/.prompts');prompts.mkdir(exist_ok=True)
mem=prompts/'MEMORY.md'
print('Current memory',mem.read_text() if mem.exists() else '(missing)')

                                               model    bar        ni       al  fe
                           Weight each study equally C70600  1.402749 3.577323 0.0
                           Weight each study equally C71500  2.369964 3.577323 0.0
           Add omitted 751d + 1064d Wheatfall blocks C70600  1.109111 3.441788 0.0
           Add omitted 751d + 1064d Wheatfall blocks C71500  2.015829 3.441788 0.0
                        Drop all Wheatfall exposures C70600  1.454913 3.599081 0.0
                        Drop all Wheatfall exposures C71500  2.433332 3.599081 0.0
Exposure-rate only, conservative exclusions (9 rows) C70600  0.000000 9.500000 0.0
Exposure-rate only, conservative exclusions (9 rows) C71500  0.000000 9.500000 0.0
               Same exclusions, no 10d Yang (7 rows) C70600 10.000000 0.000000 0.0
               Same exclusions, no 10d Yang (7 rows) C71500 30.000000 0.000000 0.0
Base 90/10 dense film rate / 70/30 1.1182017595645506
{'numpy': '2.4.3', 'pandas': '2.3

In [20]:
# Create a verified-only bibliography through the citation API (no hand-written citation file).
old_cit_path=Path('/workspace/citations/local-copper-review')/(os.environ['TRAJECTORY_ID']+'.bib')
raw=old_cit_path.read_text()
def split_bib(raw):
 out=[];i=0
 while True:
  start=raw.find('@',i)
  if start<0:break
  op=raw.find('{',start);level=1;j=op+1
  while j<len(raw) and level:
   if raw[j]=='{' and (j==0 or raw[j-1]!='\\'):level+=1
   elif raw[j]=='}' and (j==0 or raw[j-1]!='\\'):level-=1
   j+=1
  entry=raw[start:j];key=raw[op+1:raw.find(',',op)].strip();out.append((key,entry));i=j
 return dict(out)
entries=split_bib(raw)
validkeys=['ravindran1983marine','gdowski1991degradation','banning1973hotbrines','Al_Fozan_2008','Lewis_1981']
orig_traj=os.environ['TRAJECTORY_ID'];os.environ['TRAJECTORY_ID']=orig_traj+'-verified'
for key in validkeys: verified_bib=save_citation(entries[key])
os.environ['TRAJECTORY_ID']=orig_traj
print('Verified bibliography',verified_bib)
print('Four directly inspected source records:',[(k,len(entries[k])) for k in validkeys])
# Save an unambiguous executed/inspected checklist.
checklist={'independent_refit':'executed before calling supplied fitter','continuous_film_floors':'executed; separable global lower bound attained for all 32 scenarios','source_pdf_ravindran':'inspected primary PDF','source_gdowski':'inspected parsed compilation Tables 34–35','source_banning':'inspected compilation PDF Tables 7–8','source_alfozan':'anchor inspected only; direct source attempts blocked or metadata-only','source_lewis':'inspected parsed compilation Table XIV','supplied_fit_function':'executed; coefficients match to 2.84e-11','supplied_fit_main_and_grid_floor':'inspected only; not executed','full_truth':'executed with input paths overridden in memory','full_truth_floor':'executed six-start local constrained optimization in middle-Al branch; not global certification','reported_global_objective_optimum':'point re-evaluated; global search not repeated','reported_pass_shares':'inspected only; not re-sampled','calphad':'grid inspected and interpolated; thermodynamic calculation/database validation not rerun','experiments':'not executed'}
(OUT/'execution_checklist.json').write_text(json.dumps(checklist,indent=2))
mem.write_text('''# Project Memory

## Context
- For benchmark follow-up reviews, deliver a Markdown report as `outputs/report.md` and return its contents in the answer.

## Notes
- Distinguish executed checks from inspection-only checks. Include a severity-ranked findings table, independently refitted/optimized sensitivity results with witness compositions, and a short verdict.
- Keep machine-readable analysis results and a runnable reproduction script alongside the report; preserve uploaded inputs.
''')
print('Workflow memory updated; no scientific findings recorded.')

Verified bibliography /workspace/citations/local-copper-review/e8550274-e07d-4d00-9af7-1a2a7d7da0c0-verified.bib
Four directly inspected source records: [('ravindran1983marine', 321), ('gdowski1991degradation', 288), ('banning1973hotbrines', 322), ('Al_Fozan_2008', 371), ('Lewis_1981', 275)]
Workflow memory updated; no scientific findings recorded.


In [21]:
# Preserve reproduction for the executed full-truth local searches as well.
needed={'EvalTruth','full_floor'};fullblocks={}
for cell in get_ipython().history_manager.input_hist_raw:
 try:tree=ast.parse(cell)
 except SyntaxError:continue
 for node in tree.body:
  if isinstance(node,(ast.FunctionDef,ast.ClassDef)) and node.name in needed:fullblocks[node.name]=ast.get_source_segment(cell,node)
fullscript='''"""Reproduce local constrained full-truth immersion-floor searches.
Run: python outputs/reproduce_full_truth.py [input_directory]
Requires torch and botorch; does not change uploaded inputs.
These middle-Al-branch multistart solutions are NOT global certificates.
"""
from pathlib import Path
import sys,os,json,time
import numpy as np
import torch
from scipy.optimize import minimize
ROOT=Path(sys.argv[1]).resolve() if len(sys.argv)>1 else Path(__file__).resolve().parent.parent
sys.path.insert(0,str(ROOT));os.environ['BOFN_TRUTH']='toy'
import bofn_alloy as BA
import truth_phys as TP
TP.GRID_CSV=ROOT/'calphad_grid.csv';TP.FIT_JSON=ROOT/'corrosion_fit.json'
TP._GRID=None;TP._FIT=None
torch.set_num_threads(2)
truth=TP.PhysTruth(m_ref=0.)
pt=json.loads((ROOT/'phys_tables.json').read_text())
REF=['C70600','C71500']
names=['C70600 (90-10 Cu-Ni)','C71500 (70-30 Cu-Ni)']
refs={b:TP.reference_m(n) for b,n in zip(REF,names)}
'''
fullscript+='\n\n'.join(fullblocks[n] for n in ['EvalTruth','full_floor'])
fullscript+='''\nresult={b:full_floor(refs[b][0],starts=6) for b in REF}
for b in REF:
 c=torch.tensor(BA.REFERENCE[names[REF.index(b)]])
 dense=float(TP.M_LEVEL-TP.SLOPE*TP.ln_rate(c))
 result[b+'_ideal_dense_ref']=full_floor(dense,starts=3)
for label,bm,pm in [('No beta penalty',0,1),('Double beta penalty',2,1),('No porosity penalty',1,0),('Double porosity penalty',1,2)]:
 result[label]={b:full_floor(refs[b][0],starts=3,beta_mult=bm,porosity_mult=pm) for b in REF}
out=ROOT/'outputs'/'reproduction';out.mkdir(parents=True,exist_ok=True)
(out/'full_truth_optimizations.json').write_text(json.dumps(result,indent=2))
print(json.dumps(result,indent=2))
'''
(OUT/'reproduce_full_truth.py').write_text(fullscript)
p=subprocess.run([sys.executable,str(OUT/'reproduce_full_truth.py')],capture_output=True,text=True,timeout=300)
print('Full-truth reproduction exit',p.returncode,'stdout chars',len(p.stdout),'stderr',p.stderr[-1000:])
if p.returncode==0:
 check=json.loads((OUT/'reproduction/full_truth_optimizations.json').read_text())
 print('Full-truth original-vs-script Ni difference',[check[b]['ni']-fullresults[b]['ni'] for b in REF])
# Verify requested small experiment is inside computational search limits.
for ni in [0,2.5,5]:print('Proposed test alloy',ni,3.,1.5,0,'liquidus',float(L([[ni,3.,1.5,0]])[0]),'margins',margins([ni,3.,1.5]))

Full-truth reproduction exit 0 stdout chars 9391 stderr 
Full-truth original-vs-script Ni difference [np.float64(0.0), np.float64(0.0)]
Proposed test alloy 0 3.0 1.5 0 liquidus 1082.6156140350877 margins [ 30.5        167.38438596 100.         100.        ]
Proposed test alloy 2.5 3.0 1.5 0 liquidus 1089.5466605303889 margins [ 28.         160.45333947 100.         100.        ]
Proposed test alloy 5 3.0 1.5 0 liquidus 1096.936840545609 margins [ 25.5        153.06315945 100.         100.        ]


In [22]:
# Build the complete report from executed results; tables retain extractable numeric values.
table=['| Independent refit | n | RMSE ln | Ni floor: 90–10 | Ni floor: 70–30 | Witness Al | Witness Fe |','|---|---:|---:|---:|---:|---:|---:|']
labels={
'Drop hot brine + both Wheatfall blocks':'Drop hot brine + deep/localized Wheatfall',
'Drop hot/deep/local + unspecified bronze + Zhao':'Previous exclusions + unspecified bronzes + Zhao',
'Exposure-rate only, conservative exclusions (9 rows)':'Exposure rates only, conservative exclusions†',
'Same exclusions, no 10d Yang (7 rows)':'Same exclusions, also drop 10-day Yang†'}
for r in results:
 c=r['C70600']['witness']
 table.append(f"| {labels.get(r['model'],r['model'])} | {r['n']} | {r['rmse_ln']:.3f} | {r['C70600']['floor']:.3f} | {r['C71500']['floor']:.3f} | {c[1]:.3f} | {c[2]:.3f} |")
refit_table='\n'.join(table)
report=r'''# Follow-up review: does the refitted copper-alloy benchmark establish a nickel floor?

## Verdict

**No: I would not restrict the search to Ni ≥10 wt%.** The new copper anchors support a protective effect of nickel in several tested environments, but they do not establish a universal minimum Ni content for Cu–Ni–Al–Fe–Mn deposited by directed energy deposition (DED), particularly at **30 days**. There is still no measured 0–10 wt% Ni composition series that separates Ni from Al, Fe, microstructure and exposure history.

My independent base refit gives globally optimized **film-only** floors of **1.509 wt% Ni against C70600 (90–10)** and **2.500 wt% against C71500 (70–30)**, not 10 wt%. Changing quadratic Al to linear Al raises those numbers to **10 and 30 wt%**, respectively. That is the largest single model-form effect in the requested sensitivities. These are conditional model boundaries, **not experimentally established exclusion limits or confidence bounds**.

In the supplied full truth, I independently find feasible deposited-alloy boundary solutions at **2.205 and 3.264 wt% Ni**. These are multistart local optimization results, not certified global minima. More decisively, an executed witness with **Ni 5.000, Al 2.907, Fe 3.942, Mn 0, Cu 88.151 wt%** has immersion index **0.88113**, above both implemented bars (**0.81049 and 0.84207**), and satisfies every search-space limit. A Ni ≥10 restriction therefore discards passing alloys **even under the benchmark's own truth**.

**Use dense C70600 as the primary meaning of “no worse than the reference,” because that is the stated default.** Report C71500 as a separate, explicitly stricter requirement—not as a retrospective replacement chosen to obtain a desired floor. Both references must be tested alongside candidates under the actual 30-day protocol.

## 1. Severity-ranked findings

| Severity | Finding | Consequence | Evidence/check status |
|---|---|---|---|
| **Changes the conclusion** | The Al response controls whether a low-Ni route exists: quadratic versus linear Al changes floors from 1.509/2.500 to 10/30 wt% Ni. | A universal Ni floor, especially 10 wt%, is not robust to defensible functional forms. | **Executed:** independent refits and continuous optimization. |
| **Changes the conclusion** | Strict exclusions leave the Al response unidentified: the nine-row exposure-rate subset has within-study feature rank 3, and removing the 10-day Yang pair leaves rank 2 of 4. | Numerically generated floors of 0/0 or 10/30 in those subsets reflect ridge extrapolation or zero-filled coefficients, not evidence resolving the floor. | **Executed:** exclusions, rank checks, refits, optimized witnesses. |
| **Changes the conclusion** | None of the 14 additions measures 30-day low-Ni DED performance; several are hot brine, deep ocean or localized-loss tests. | Same-test ratios cannot by themselves transport alloy effects to 30-day ambient immersion. | **Inspected:** primary/compilation tables for 12 rows; uploaded anchor only for the last two. |
| **Changes the conclusion** | The full truth admits a clear 5 wt% Ni passing witness. Its Ni ~12 wt% objective optimum also rewards strength, processability and mineral burden. | Neither that objective optimum nor the sampled passing fraction establishes a minimum Ni requirement. | **Executed:** witness, feasibility and optimum-point evaluation; global objective search not repeated. |
| **Changes a number** | The shipped “floor” routine uses a 0.1 wt% grid, not continuous optimization. | Its base 1.6/2.6 becomes 1.509/2.500; its first passing witness is not the optimal Al witness. | **Inspected:** routine; **executed:** independent optimization. |
| **Changes a number** | “Dense” truth references are actually optimized DED references with density indices 0.98024/0.97356. | Their porosity penalties relax the bar relative to truly dense references; corrected full-truth local boundaries are 2.387/3.569 wt% Ni. | **Executed:** reference calculation and constrained reoptimization. |
| **Changes a number** | The full-truth boundary witnesses in the upload are not minima I recover. | Their 2.246/3.293 wt% becomes 2.205/3.264 with slightly better process/composition choices. | **Executed:** six-start constrained optimization in the middle-Al branch. |
| **Changes a number** | Two matched Wheatfall exposure pairs in the anchor were omitted, including the 1,064-day pair where copper has lower loss. | Including both changes film floors to 1.109/2.016; the additions should not be presented as exhaustive evidence. | **Inspected:** Tables 34–35; **executed:** added-pair sensitivity. |
| **Changes a number** | Nominal/assumed compositions, electrochemical proxies, generic bronze assignments and hand-written penalties are treated as if they determine physical rate ratios. | Numerical precision overstates calibration quality; model-form and composition uncertainty dominate. | **Inspected:** data and equations; **executed:** exclusions and penalty sensitivities. |
| **Changes a number** | Mn =0 does not generally minimize grid liquidus, contrary to the fitting helper's comment. | It is not a valid general reduction of the feasible domain. It does not alter my film floors because each unrestricted lower-bound witness is already feasible at Mn =0. | **Executed:** grid-node comparisons and witness feasibility. |
| **Cosmetic / reproducibility** | The uploaded files are flat, but `truth_phys.py` expects `phys/`; reference compositions in the fitter and truth differ slightly. Ravindran has a publisher-listed DOI absent from the anchor. | Fix packaging and use one reference-composition definition. The DOI endpoint returned 404, so I saved verified publisher metadata instead. | **Executed:** path-only overrides, composition comparison, metadata retrieval. |

## 2. Audit of the 14 added rows

CSV line numbers below include the header. I checked **all 14 against `copper-anchor.md`**. The numerical rates agree with it. Agreement with that narrative is not independent source verification; I distinguish that explicitly below. The nominal/assumed composition columns are not equivalent to measured specimen chemistry.

| CSV lines / rows | Source comparison | Assessment and appropriate use | Check performed |
|---|---|---|---|
| **18–20:** Cu 61.0; 90–10 22.5; 70–30 22.5 µm/y | Ravindran 1983, Table 5, PDF p.5 / printed p.94 [R1]. All three panels shared 120 days at Cochin Harbour, with tidal flushing, still to 0.80 m/s. | **Fair internally matched field comparison**, not quiet 30-day immersion. The method for Table 5 is not explicitly established as weight loss. Ni 10/30 is nominal; the CSV's Fe 1.4/0.6 and Mn 0.8/0.6 are assumptions, not Table 5 assays. Do not confuse the paper's separately discussed weight-loss curves with confirmation of this table's method. | **Primary PDF inspected; rates verified.** |
| **21–22:** Cu 12.1; Cu–30Ni 6.27 mdd | Wheatfall 1967 via Gdowski 1991, Tables 34–35, PDF p.45 / printed p.40 [R2]. Matched 123-day exposure at 5,640 ft, 2.78°C, ~2,500 psi, dissolved O₂ 1.19 ppm. | **Fair same-exposure mass-loss pair, weak target-domain anchor.** Cold, low-oxygen deep ocean is not ambient surface immersion. Cu–Ni Fe/Mn are assumed. | **Compilation inspected; original Wheatfall experiment not retrieved.** |
| **23–24:** Cu 4.50; Cu–30Ni 1.42 mdd | Same tables [R2]; surface, 386 days, 17.61°C, 1–2 knots. Footnote b explicitly attributes the copper loss largely to localized corrosion. | **Matched exposure, but not a fair uniform-film-rate calibration.** It can inform a localized-damage endpoint, provided that endpoint is separately defined. Drop the pair from a uniform-film sensitivity—not just the copper row, which would leave no within-block contrast. | **Compilation inspected; footnote verified.** |
| **25–29:** C122 0.99; C608 0.75; C613 0.73; C706 0.70; C715 0.46 mpy | Cohen–Whitted 1971 via Banning–Oden 1973, Tables 7–8, PDF pp.18–19 / printed pp.14–15 [R3]. These are genuinely the same **4 ft/s, 697-day** column. | **Fair same-loop comparison, not fair direct transport to the target test.** Temperature 250°F (~121°C), 83% normal seawater salinity, treated oxygen-poor brine, 72 ppb O₂ in makeup. Table 7 gives specification maxima/ranges, not assays. C608 Al 5.75 and Fe 0.05 are assumed representatives; C613 Al 7 and especially Fe 2.5 are not measured values. C613's specified 0.2–0.5% Sn is omitted from the model. C715 Ni 31 and Fe 0.55 represent ranges, not measured chemistry. | **Compilation PDF inspected; all five rates and column match verified. Original Corrosion/71 paper not retrieved.** |
| **30–31:** 90–10 0.145; 70–30 0.067 mpy | Al-Fozan–Malik 2008, Tables 1 and 3, as transcribed in the anchor [R4]. Nominally same immersion conditions, but **198 versus 192 days**. | **Not an exact same-duration pair.** The measured compositions in the CSV agree with the anchor; the nearby durations make this a reasonable qualified sensitivity input, not a 30-day measurement. It adds a 10-versus-30 comparison, not a low-Ni anchor. | **Anchor inspected only for numbers.** DOI metadata retrieved; full-text retrieval failed/was blocked or returned metadata only. I did **not** independently verify its tables. |

**Omitted evidence:** Gdowski also reports Cu/Cu–30Ni **3.78/2.75 mdd at 751 days** and **2.65/3.08 mdd at 1,064 days**. The latter reverses the copper-versus-Cu–Ni ordering. I added both as separate exposure blocks in a sensitivity, not as extra independent studies. The uploaded “11 studies” are actually **11 fitted blocks**, with Wheatfall split by exposure and Lewis–LaQue appearing twice; that is not 11 wholly independent sources.

**Older-row issue relevant to this refit:** Lewis–LaQue Table XIV [R5] really reports uncoupled aluminum bronze **43.2**, nickel aluminum bronze **150.0**, C70600 **30.5** and C71500 **20.3 µm/y** after one year at 0.6 m/s. I verified those values, but the table does **not** establish the particular C61400/C95800 compositions assigned in the CSV. Those assumed compositions are consequential Al-response inputs, not measured low-Ni bronze chemistry.

## 3. Independent refit and optimized floors

### Method and verification

I wrote and executed my own fitting code **before calling the supplied fitter**. I eliminated each study/exposure intercept by **weighted within-block centering**, then solved the bounded ridge regression on the four composition features. This is an independent implementation of the stated weighted least-squares objective, not a fit to the supplied coefficients.

For the base case, with Al² divided by 10:

\[
(b_{Ni},b_{Al},b_{Al2},b_{Fe})
=(-0.840921,-0.279938,+0.385670,\;\text{approximately }0).
\]

Unweighted residual root-mean-square error (RMSE) is **0.226696 in ln(rate)** for 30 rates. The independent coefficients match the subsequently executed supplied fit function to **2.84×10⁻¹¹** maximum absolute difference. This confirms implementation agreement, not physical validity. In particular, the protective Fe coefficient is on its imposed zero boundary; the unconstrained fit instead estimates **+0.12880**. Freeing signs is a diagnostic of confounding/model inadequacy, not proof that Fe is intrinsically harmful.

I minimized Ni continuously over Ni 0–30, Al 0–9.5 and Fe 0–4.5 wt%, with Cu ≥65%, the two conditional composition rules and interpolated liquidus ≤1,250°C. For the film model, I first obtained the exact unrestricted lower bound by minimizing the separable Al and Fe contributions and solving the Ni constraint. **Every one of the 32 fits below attains that lower bound at a feasible witness with Mn =0.** Thus these film-model floors are globally certified for the stated deterministic model, not simply grid hits or local minima. Allowing Mn to vary cannot beat an already attained unrestricted lower bound.

All witnesses have Cu as the balance. The table gives common witness Al/Fe for both bars; combine those columns with the corresponding Ni floor and Mn =0. Values are rounded for display; exact feasible compositions and residual margins are in `optimized_witnesses.csv`. Film comparisons use the supplied fitter's reference compositions **(Ni,Al,Fe) =(10,0,1.4)** and **(30,0,0.6)**.

__REFIT_TABLE__

† **Do not interpret the last two rows as well-identified alternatives.** The nine-row subset retains Yang, the two Lewis–LaQue Cu–Ni rows, Ravindran and Al-Fozan. Its sole Al contrast is the **10-day Yang** pair; linear and quadratic Al are not separately identifiable. Ridge selects an extrapolated high-Al optimum, giving zero-Ni witnesses at Al 9.5. Without Yang, no Al contrast remains at all; ridge sets both Al coefficients to zero and recovers 10/30 by construction. These rows expose an evidence gap, not a credible measured 0% or 10% cutoff.

### What moves the floor?

- **Largest requested single model-form change: Al linear versus quadratic.** Floors shift **+8.491 and +27.500 wt% Ni** from the base case. In the quadratic base fit, Al **3.629 wt%** lowers log rate by **0.508**, a rate multiplier of **0.602**. This inferred benefit opens the low-Ni route; it is not directly measured in a 30-day low-Ni series.
- **Ni response shape matters**, especially for the stricter bar: scales 3/5/8/12 give 70–30 floors **1.311/2.500/4.341/5.787**; linear Ni gives **7.822**. Linear Ni has lower training RMSE (**0.184 versus 0.227**), so the data do not select the assumed 5 wt% saturation scale by fit quality. Training RMSE alone is not model validation.
- **Signs matter:** freeing them lowers the quadratic floors to **0.406/1.956**; linear Al with free signs gives **4.603/10.807**. The newly anchored free-sign fits do not make Ni harmful; the earlier behavior is not reproduced here.
- **Study selection matters but does not recover a robust 10% rule.** Dropping hot brine raises floors to **2.896/4.249**; dropping Tan lowers them to **0/0.568**. Those are the largest leave-one-block shifts for the 70–30 bar. Dropping all hot/deep/localized additions gives **2.889/4.240**.
- **Weights and weak ridge are smaller effects here.** Equal row weights give **0.918/1.787**; equal total weight per fitted block gives **1.403/2.370**. Removing or strengthening ridge changes floors by <0.19 wt% from base. Fe-cap changes have no effect in the constrained base fit because its Fe coefficient is zero.

These are **sensitivity envelopes, not statistical confidence intervals**. There are too few comparable independent tests, and too much uncertain chemistry and endpoint mixing, to attach an honest narrow confidence interval to a physical floor.

## 4. Immersion physics and use inside the truth

### Executed full-truth checks

I ran `PhysTruth` after overriding **only its two input paths in memory**, because the uploaded layout is flat. I did not edit any uploaded input. I reproduced both implemented reference indices and the uploaded floor witnesses. I then minimized Ni with six-start constrained sequential least-squares programming over composition and the three process settings, in the **1.5–7 wt% Al branch**. All six starts converged to the same reported boundary within numerical tolerance. Other branches/global optimality were **not certified**.

| Reference convention | Bar | Best Ni found | Al | Fe | Mn | Density index | Beta index |
|---|---|---:|---:|---:|---:|---:|---:|
| Implemented best-DED reference | C70600 | 2.205 | 2.907 | 3.942 | 0 | 0.98523 | 0.10758 |
| Implemented best-DED reference | C71500 | 3.264 | 2.913 | 3.984 | 0 | 0.98632 | 0.10639 |
| Ideal dense reference, candidate still DED | C70600 | 2.387 | — | — | — | — | — |
| Ideal dense reference, candidate still DED | C71500 | 3.569 | — | — | — | — | — |

Exact compositions/process settings for every row are saved in `full_truth_optimizations.json`. Threshold equalities are numerical, not experimental pass margins. The separate **5 wt% Ni witness in the verdict** has a clear positive margin instead.

The ideal dense-reference indices are **0.816415/0.850000**, versus the implemented **0.810487/0.842067**. The model currently calls a porosity-penalized deposited reference “dense.” Also, `bofn_alloy.REFERENCE` actually decodes to C70600 **Ni 9.99, Fe 1.395, Mn 0.8** and C71500 **Ni 30, Fe 0.675, Mn 0.8**, not exactly the fitter's reference compositions. This discrepancy is small for the base fit, where Fe has zero effect, but should be removed before sensitivity comparisons.

Keeping the implemented bar fixed while changing **candidate-only** hand-written immersion penalties gives:

| Candidate penalty sensitivity | Ni floor found, 90–10 | Ni floor found, 70–30 |
|---|---:|---:|
| Baseline | 2.205 | 3.264 |
| No beta-prime penalty | 1.442 | 2.348 |
| Twice beta-prime penalty | 2.712 | 3.892 |
| No porosity penalty | 1.358 | 2.255 |
| Twice porosity penalty | 2.323 | 3.397 |

This is a perturbation diagnostic, **not a refit or recalibration of reference bars**.

### Physically misleading assumptions

1. **No 30-day kinetic model.** Study intercepts remove absolute levels and units, not alloy-by-time, temperature, oxygen, flow or salinity interactions. Rates from 7-day impedance, 10-day mass loss, 30-day polarization and up to 697-day brine exposure cannot automatically share a transferable 30-day alloy response. The model predicts relative surrogate rates; it has not calibrated an absolute 30-day mass loss.
2. **Mixed observables are not interchangeable rates.** A polarization current or inverse charge-transfer resistance can have alloy-dependent conversion factors and localized-attack sensitivity. A study intercept does not remove these within-study differences. Midpoints of broad literature rate ranges also are not a matched exposure experiment.
3. **Film coefficients and signs are assumptions, not a floor mechanism.** Saturating Ni at a fixed 5 wt% scale, clipping Fe at 1.5 wt%, and requiring nonpositive Ni/Fe slopes encode prior beliefs. `min(nominal Fe,1.5)` is not measured or CALPHAD-derived **Fe in solution**. Mn, missing minor elements and microstructure are omitted from immersion chemistry.
4. **The Al quadratic conflates film chemistry and microstructure.** The source mixtures already contain cast/duplex/laser-deposited alloys with their actual phases and defects. Adding a separate hand-written beta-prime penalty risks double-counting selective attack. It is particularly misleading to read the beta index as a phase fraction: the formula produces ~0.108 at only ~2.9 wt% Al without any phase-existence check, and it is not tied to the supplied CALPHAD beta field. This is a synthetic index, not demonstrated retained beta-prime in those witnesses.
5. **Porosity treatment is not fitted corrosion physics.** A linear penalty in `1−rho_m` cannot represent connected versus closed pores, crevices, exposed area or localized penetration. The “density” node also combines process window and cracking into an index, so it is not an independently measured pore-volume fraction. Translating the penalty into log-rate units does not validate it.
6. **Implementation choices affect benchmark conclusions.** The immersion output is clipped to [0,1], so different rates can collapse to the same index near saturation. The objective uses a **soft sigmoid gate**, not a hard pass requirement; an objective optimum is not the minimum-Ni passing alloy. The one-hour screen is hand-written and ignored by the actual immersion truth despite being a parent input to its learned surrogate. Any claimed screen/immersion discrepancy remains partly designed into the benchmark.
7. **CALPHAD is not validation of DED phase selection.** I inspected/interpolated the supplied grid, not reran pycalphad or validated the database. Equilibrium fields and multilinear interpolation do not establish retained phases in rapidly cooled, segregated DED material. All 864 rows have `ok=1`; 33 missing liquidus entries are replaced in the truth with 1,550°C scan-bound values, which are not measured liquidus temperatures. None of my low-floor witnesses is near that substitution region or the 1,250°C limit.

The benchmark can still be useful as a **documented synthetic stress test**. It should not be described as experimentally established physical truth that excludes low-Ni alloy families.

## 5. Smallest discriminating experiment

**To test the proposed Ni ≥10 restriction, the smallest useful falsification experiment is one low-Ni candidate plus both references:**

- DED **Cu–5Ni–3Al–1.5Fe**, Mn nominally zero, Cu balance **90.5 wt%**. This tests the predicted Al-assisted low-Ni route, not pure Cu–Ni extrapolation.
- Dense, assayed **C70600** and **C71500**, separately identified as the default and stricter comparators.
- Start with **three independently deposited candidate specimens** and three independent specimens of each reference: **nine coupons**. This is a minimum screening design, **not a power-justified noninferiority trial**. Independently deposited specimens/build positions matter; repeatedly weighing one coupon is not replication.

**To distinguish a ~0%, ~2–3% and ≥5% transition, add Cu–0Ni–3Al–1.5Fe and Cu–2.5Ni–3Al–1.5Fe**, with the same replication: **five alloy groups, 15 coupons**. These three candidate compositions all satisfy the computational search limits. An optional unalloyed-copper group is a useful process/chemistry sentinel but is not essential to the first falsification.

Use **30 days of simultaneous, fully submerged, electrically isolated immersion** in one specified seawater batch/protocol, preferably clean aerated natural seawater at **25 ±2°C** with controlled gentle circulation (or explicitly specified static immersion), measured dissolved oxygen, salinity and pH. Match exposed area and final surface finish; control/record renewal, film preconditioning and edge/crevice geometry. Use an appropriate standardized immersion/cleaning procedure, cleaning blanks and area-normalized mass loss, plus maximum pit depth and cross-sectional selective attack. Assay Ni/Al/Fe/Mn, characterize phases and connected porosity, and confirm the low-Ni deposits are not diluted substrate mixtures.

Pre-register what “no worse” means: the **30-day cumulative mass-loss ratio to C70600**, with C71500 secondary, and a separate localized-damage requirement. For a literal ratio ≤1 criterion, evaluate a prespecified one-sided upper confidence bound; with only three replicates it may be inconclusive. Set a practically acceptable noninferiority margin *before* seeing results if literal equality is experimentally unrealistic, and then increase replication based on pilot variability. A passing 5% alloy falsifies a universal ≥10% floor; a failed alloy does **not** prove one. No finite small experiment proves that every lower-Ni composition fails. A 90-day follow-on is useful for film maturation, but should not replace the specified 30-day decision.

## 6. What I executed versus only inspected

| Check | Status |
|---|---|
| Independent base and 31 alternative refits; continuous film-floor optimization; full witness feasibility | **Executed**, with exact lower-bound attainment certificates for the deterministic film model. |
| Supplied `fit_corrosion.fit` coefficient comparison | **Executed after my implementation**. Supplied `main()` and its gridded floor calculation were **inspected, not rerun**. |
| All 14 additions against uploaded anchor | **Inspected all rows**; 12 values additionally checked in directly accessible primary/compilation tables; Al-Fozan's two table values **not independently source-verified**. |
| Full-truth reference calculation, uploaded witnesses and optimum point, new multistart floor searches, dense-bar and penalty sensitivities | **Executed**. Full-truth global minima and the uploaded global objective optimum **not independently certified**. |
| Uploaded 5.2%/3.2% passing shares | **Inspected only**, not resampled; these depend on the sampling distribution, not just the alloy domain. |
| CALPHAD grid completeness, liquidus interpolation and Mn dependence | **Executed grid checks**. Underlying thermodynamic calculations/database and real DED phases **not validated**. |
| Reproduction scripts | **Both executed successfully in fresh Python subprocesses**; reproduced recorded floor values exactly in this environment. No experimental work performed. |

## 7. Files and sources

Saved alongside this report:

- `refits.csv`, `refits.json`: coefficients, fit errors and optimized floors for all 32 scenarios.
- `optimized_witnesses.csv`: full Ni/Al/Fe/Mn/Cu witnesses, liquidus and rate/feasibility margins for both bars.
- `new_rows_audit.csv`: all 14 row-level provenance checks and qualifications.
- `full_truth_optimizations.json`: exact composition/process witnesses and penalty/dense-reference sensitivities.
- `reproduce_refits.py`, `reproduce_full_truth.py`, `environment.json`, `execution_checklist.json`; downloaded/parsed evidence in `sources/`. Uploaded inputs remain unchanged.

Reproduce from `/workspace`:

```bash
python outputs/reproduce_refits.py
python outputs/reproduce_full_truth.py
```

The second script requires PyTorch and BoTorch; exact installed versions are recorded. Source retrieval need not be repeated for the numerical reproduction.

**Source references** — full BibTeX records are saved in the [verified bibliography](../citations/local-copper-review/e8550274-e07d-4d00-9af7-1a2a7d7da0c0-verified.bib).

- **[R1; `ravindran1983marine`]** K. Ravindran (1983), “Copper-Nickel Alloys for Marine Aquacultural Engineering,” *Fishery Technology* **20**(2), 90–96, Table 5. [Primary PDF](https://epubs.icar.org.in/index.php/FT/article/download/70952/60283/468475). Publisher-listed DOI **10.56093/ft.v20i2.70952**; resolver returned 404 during this review.
- **[R2; `gdowski1991degradation`]** G. E. Gdowski (1991), *Survey of Degradation Modes of Four Nickel-Chromium-Molybdenum Alloys*, Tables 34–35, compiling Wheatfall (1967). DOI: [10.2172/138275](https://doi.org/10.2172/138275).
- **[R3; `banning1973hotbrines`]** L. H. Banning and L. L. Oden (1973), *Corrosion Resistance of Metals in Hot Brines: A Literature Review*, U.S. Bureau of Mines IC 8601, Tables 7–8, compiling Cohen and Whitted, Corrosion/71 Paper 57. [Compilation PDF](https://stacks.cdc.gov/view/cdc/235753/cdc_235753_DS1.pdf).
- **[R4; `Al_Fozan_2008`]** S. A. Al-Fozan and A. U. Malik (2008), “Effect of seawater level on corrosion behavior of different alloys,” *Desalination* **228**, 61–67. DOI: [10.1016/j.desal.2007.08.007](https://doi.org/10.1016/j.desal.2007.08.007). Numerical table claims here are attributed to the uploaded anchor, **not independently retrieved full text**.
- **[R5; `Lewis_1981`]** R. O. Lewis and F. LaQue (1981), *Evaluation of Copper-Nickel Alloys for OTEC Applications*, Table XIV, compiling “Inco Unpublished Data.” DOI: [10.2172/6255887](https://doi.org/10.2172/6255887).

## Discretionary analytical decisions

- I retained the supplied log-rate/ridge objective and weights for the base case, so the first check isolates implementation correctness rather than silently changing the model.
- I used exact separable optimization with feasibility verification instead of a denser Ni grid; it certifies the film-model floors without grid-rounding ambiguity.
- I removed entire two-row exposure blocks when their comparisons were weak, and retained the two source-reported omitted Wheatfall pairs in a separate sensitivity. I did not select only favorable copper-anchor results.
- I separated plausible model-form sensitivities from rank-deficient strict-data fits, and did not treat synthetic sensitivity ranges as physical confidence intervals.
- I kept C70600 primary, C71500 secondary, and distinguished ideal dense references from optimized deposited references. I labeled full-truth searches as local rather than claiming a global result I did not establish.
- I recommended the smallest falsification study separately from a larger transition-mapping study; neither can prove a universal minimum over an alloy continuum.
'''
report=report.replace('__REFIT_TABLE__',refit_table)
(OUT/'report.md').write_text(report)
print('Report saved:',OUT/'report.md','words',len(report.split()),'characters',len(report))
print('Report refit table:')
print(refit_table)
# Save hashes so preservation and reruns can be checked.
import hashlib
inputs=['truth_phys.py','corrosion_data.csv','fit_corrosion.py','corrosion_fit.json','corrosion_fit.md','phys_tables.md','phys_tables.json','copper-anchor.md','calphad_grid.csv','bofn_alloy.py']
(OUT/'input_sha256.json').write_text(json.dumps({f:hashlib.sha256(Path('/workspace',f).read_bytes()).hexdigest() for f in inputs},indent=2))

Report saved: /workspace/outputs/report.md words 4519 characters 30411
Report refit table:
| Independent refit | n | RMSE ln | Ni floor: 90–10 | Ni floor: 70–30 | Witness Al | Witness Fe |
|---|---:|---:|---:|---:|---:|---:|
| Base | 30 | 0.227 | 1.509 | 2.500 | 3.629 | 0.000 |
| Linear Al | 30 | 0.323 | 10.000 | 30.000 | 0.000 | 0.000 |
| Signs free | 30 | 0.223 | 0.406 | 1.956 | 3.752 | 0.000 |
| Linear Al, signs free | 30 | 0.320 | 4.603 | 10.807 | 0.000 | 0.000 |
| Ni scale 3 | 30 | 0.249 | 1.150 | 1.311 | 3.501 | 0.000 |
| Ni scale 8 | 30 | 0.204 | 1.355 | 4.341 | 3.691 | 0.000 |
| Ni scale 12 | 30 | 0.193 | 0.669 | 5.787 | 3.650 | 1.500 |
| Linear Ni | 30 | 0.184 | 0.000 | 7.822 | 3.418 | 1.500 |
| Fe cap 1.0 | 30 | 0.227 | 1.509 | 2.500 | 3.629 | 0.000 |
| Fe cap 2.5 | 30 | 0.227 | 1.509 | 2.500 | 3.629 | 0.000 |
| No ridge | 30 | 0.227 | 1.496 | 2.484 | 3.666 | 0.000 |
| Ridge 0.5 | 30 | 0.234 | 1.659 | 2.683 | 3.340 | 0.000 |
| Equal row weights | 30 | 0.224 | 0.918 | 1.787 | 

898

In [23]:
# Final integrity checks on the saved report and reproduction artifacts.
report_path=OUT/'report.md';saved_report=report_path.read_text()
assert '__REFIT_TABLE__' not in saved_report
assert saved_report.count('| Base |')==1
assert len(pd.read_csv(OUT/'new_rows_audit.csv'))==14
assert len(pd.read_csv(OUT/'optimized_witnesses.csv'))==64
assert len(json.loads((OUT/'reproduction/refits.json').read_text()))==32
for f in ['reproduce_refits.py','reproduce_full_truth.py']:
 compile((OUT/f).read_text(),str(OUT/f),'exec')
print('PASS: report saved; 14 audited additions; 32 fits; 64 feasible film witnesses; both scripts compile and were run successfully.')
print('Report:',str(report_path))
print('Disk usage:',subprocess.run(['du','-sh',str(OUT)],capture_output=True,text=True).stdout.strip())

PASS: report saved; 14 audited additions; 32 fits; 64 feasible film witnesses; both scripts compile and were run successfully.
Report: /workspace/outputs/report.md
Disk usage: 4.1M	/workspace/outputs
